# PR01 · Read an image without losing its location

**Format:** 75–100 minutes for this lesson and its small executable lab, followed by the explicitly labeled upstream practice. Run this notebook from a fresh kernel, top to bottom. Core data are synthetic. Software: NumPy, SciPy, Matplotlib; extra imports are stated in code. This is one stage of a longer processing course, not a replacement for supervised research training.

## Learning objectives

- Identify data, affine, voxel units and timing metadata separately.
- Preserve physical correspondence during an exact reorientation.
- Explain why a file extension and plausible brain picture are incomplete QC.

## Understand the operation

An MRI image is a numerical array accompanied by a description of its sampling. The number at index `(i,j,k)` is not a location until a coordinate mapping is attached. A 4×4 affine maps homogeneous voxel coordinates to world coordinates. Its first three columns describe index-axis directions and spacing; its last column is the origin. The determinant gives signed voxel volume, while its absolute value gives physical volume for the spatial linear mapping. A negative determinant is a handedness fact, not proof of erroneous data.

A NIfTI header can contain qform and sform mappings. Their codes and consistency matter; selecting a preferred affine is a software decision you should record. Orientation labels alone do not tell you whether two brains are registered. Likewise, a fourth array dimension may index diffusion encodings rather than evenly sampled fMRI time. NIfTI units and BIDS sidecars carry information that array shape cannot recover. DICOM conversion is an earlier acquisition-specific task; record the converter and retain authorized source metadata.

This lab reverses one index axis and updates its affine. The intensity multiset stays exactly the same while addresses change. We also create and reload a tiny NIfTI image in a temporary directory. This tests actual file I/O without distributing a person's scan. Nothing in these operations reconstructs a missing orientation label or establishes anatomical correctness.

## Transformation contract

**Input:** float32 `(12,14,10)` values and a millimeter affine. **Output:** a NIfTI object and a reindexed equivalent. **Preserved:** samples and world locations when index map and affine agree. **Lost:** nothing during exact reversal; interpolation is not involved. **Inspect:** dtype, shape, units, qform/sform codes and selected world coordinate.


## Read the actual course material

- [Berkeley Psych214: imaging data and transforms](https://github.com/bic-berkeley/psych-214-fall-2016/blob/ec44652addc92091183456fa04fcd41dca0e172e/syllabus.rst). CC BY 2.0 for website text; linked only.
- [Nipype tutorial: BIDS inputs](https://github.com/nipy/nipype_tutorial/blob/f11c9c7b8e7a1983918f1d517ec9cf3dfcb78236/notebooks/basic_data_input_bids.ipynb). BSD-3-Clause; unmodified upstream copy in third_party/processing.

Read the named topic alongside this lesson; compare its real-image assumptions with our controlled example. These notebooks use original explanations and original code, not copied upstream passages. The source chapter is the place to continue to a complete real-tool practical. External software and downloaded datasets are not silently run by this notebook.


## Predict, then ask your AI assistant

Use Goose with your installed Ollama model, or ChatGPT. The model is a tutor and code author; the local Python runtime performs these calculations. Paste:

> Explain how `A` maps one address to millimeters. Audit the NIfTI round trip and propose a world-coordinate assertion for an axis reversal. Do not call it registration or infer anatomical left from a plot. Return at most 20 executable lines per cell, show units and array shapes, and preserve the original. Explain the prediction before running. If an assertion fails, diagnose the disagreement rather than deleting the check.

Write your prediction before executing the reference cells below.


In [1]:
import numpy as np
import nibabel as nib
from pathlib import Path
from tempfile import TemporaryDirectory
values = np.arange(12*14*10,dtype=np.float32).reshape(12,14,10)
A = np.array([[-2,0,0,20],[0,2,0,-12],[0,0,3,-9],[0,0,0,1.]])
img = nib.Nifti1Image(values,A)
img.header.set_xyzt_units('mm','sec')
img.set_qform(A,code=1); img.set_sform(A,code=1)
with TemporaryDirectory() as folder:
    path = Path(folder)/'synthetic.nii.gz'; nib.save(img,path)
    loaded = nib.load(path); roundtrip = loaded.get_fdata()
    print('shape, units, codes:',loaded.shape,loaded.header.get_xyzt_units(),
          int(loaded.header['qform_code']),int(loaded.header['sform_code']))
assert np.array_equal(roundtrip,values)
print('voxel volume mm^3:',abs(np.linalg.det(A[:3,:3])))


shape, units, codes: (12, 14, 10) ('mm', 'sec') 1 1
voxel volume mm^3: 12.0


In [2]:
point = np.array([2,4,3,1.]); world = A @ point
F = np.eye(4); F[0,0] = -1; F[0,3] = values.shape[0]-1
reindexed = values[::-1]; A_new = A @ F
new_point = np.array([9,4,3,1.])
correct = A_new @ new_point; wrong = A @ new_point
print('original, correct, wrong world:',world[:3],correct[:3],wrong[:3])
assert np.allclose(world,correct) and not np.allclose(world,wrong)
assert np.array_equal(np.sort(values.ravel()),np.sort(reindexed.ravel()))
assert np.isclose(abs(np.linalg.det(A[:3,:3])), 12)
print('wrong-header error mm:',np.linalg.norm(wrong[:3]-world[:3]))


original, correct, wrong world: [16. -4.  0.] [16. -4.  0.] [ 2. -4.  0.]
wrong-header error mm: 14.0


## Check and explain

The volume is 12 mm³ per voxel. Index `[2,4,3]` maps to `[16,-4,0]` mm; its reversed address `[9,4,3]` must map there too. The unchanged affine misplaces it by 14 mm. Explain why the histogram check cannot detect that mistake.

## Deliberately wrong method

The `wrong` branch reverses data while retaining the old affine. A viewer could still draw a recognizable image. Reject the claim that a valid NIfTI file proves accurate anatomy. Do not repair real headers by trying flips until a picture looks familiar.

## Transfer to an actual dataset or tool — guided assignment

Use the pinned Nipype BIDS-input chapter to identify the source image and sidecar selected for one participant. On an authorized public anatomical image, print the header units and both coded forms; record the dataset version, image path and conversion provenance. Overlay known anatomical landmarks in three planes. If forms disagree, document the disagreement and investigate its origin instead of rewriting both forms. This is a 30–45 minute inspection assignment; its file must be obtained separately.

**Submit:** a transformation card, one labeled figure or numerical result, the failed-method diagnosis, and the upstream-practice evidence. If the external exercise has not been run, mark it **not executed** and state the missing software/data; do not convert a proposed command into a claimed result.

## Exit questions and answer key

1. Why is a matching shape insufficient? **Answer:** grids can have different spacing, orientation, origins or anatomical correspondence.
2. What must accompany reindexing? **Answer:** the corresponding index-to-world mapping; voxel values alone cannot establish location.
